# 🚢 선박기기 매뉴얼 검색 PoC — 모델 선정 근거

**과제**: 내 도메인(선박기기 매뉴얼)에 AI를 적용해 실제 개선을 증명하는 PoC.

이 노트북은 **어떤 모델을, 왜 선택했는지(대안 대비)** 를 실측 결과와 함께 정리한 문서입니다.
전체 실행 코드는 저장소의 [`app.py`](app.py) 에 있습니다.

> ⚠️ 이 노트북의 코드 셀은 **설명용 발췌**이며 실행 출력은 비워 두었습니다.
> 표에 적힌 수치는 개발 중 동일 매뉴얼로 **실제 측정한 값**입니다(해당 절에 "실측"으로 표기).

## 0. 문제 정의

- **대상**: 선박기기(엔진·펌프·발전기·분석기 등) 매뉴얼 PDF
- **특성**: ① 스캔본 + 텍스트본 혼재 ② 도면·표·차트 등 **그림 위주** ③ 장비당 수백 페이지
- **기존 방식의 한계**: 스캔본은 `Ctrl+F`가 안 되고, 도면은 글자 검색으로 못 찾음 → 원하는 그림/내용 찾기가 고역
- **목표**: "도면을 말로 찾고, 본문을 한글로도 찾는" **완전 로컬** 검색기

## 1. 제약 조건 (모델 선택을 좌우한 요건)

| 제약 | 내용 | 영향 |
|---|---|---|
| **완전 로컬** | 외부 API·통신 금지 (보안망) | 오픈소스·로컬 실행 모델만 |
| **실행 환경** | Windows 10 · **Python 3.14** · RTX 3060(6GB)/CPU | py3.14 휠 없는 패키지 배제 |
| **입력 다양성** | 스캔본 + 텍스트본 | OCR 필요 |
| **콘텐츠** | 도면·표(이미지) + 본문(텍스트) | 멀티모달 |
| **질의 언어** | 한글 질의 → 영문 매뉴얼 | 교차언어 검색 |

이 제약들이 아래 모델 선정의 **결정적 근거**가 됩니다.

## 2. 모델 선정 근거

### 2-1. 이미지 임베딩 — `google/siglip2-base-patch16-224`

- **역할**: 도면·표 이미지를 벡터로 만들고, 텍스트 질의와 같은 공간에서 비교 → "말로 그림 검색"
- **선정 이유**: 이미지↔텍스트를 하나의 공간에 매핑하는 멀티모달 모델. 오픈소스·로컬 실행. CLIP 대비 최신(Sigmoid loss)으로 검색 품질이 좋고 base 크기라 6GB에서도 가벼움.
- **대안 비교**: 순수 이미지 분류/특징 모델(ResNet 등)은 "텍스트로 검색"이 불가 → 제외. CLIP도 가능하나 SigLIP2가 후속·개선판.

In [ ]:
# (발췌 — 전체는 app.py) 이미지 임베딩: SigLIP2, L2 정규화
from transformers import AutoModel, AutoProcessor
import torch

MODEL_ID = "google/siglip2-base-patch16-224"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).eval()

@torch.no_grad()
def embed_image(img):
    inputs = processor(images=[img.convert("RGB")], return_tensors="pt")
    feats = model.get_image_features(**inputs).pooler_output   # transformers 5.x
    return (feats / feats.norm(dim=-1, keepdim=True)).cpu().numpy()[0]

### 2-2. 레이아웃 검출 — `DocLayout-YOLO` (DocStructBench)

- **역할**: 페이지에서 **그림(figure)·표(table)** 영역을 검출해 잘라냄 → 페이지 통째가 아니라 도면/표만 정밀 색인
- **선정 이유**: 문서 레이아웃 검출용으로 **사전학습**되어 있어 별도 학습 불필요. `figure`/`table` 클래스를 바로 제공.
- **대안 비교**: 직접 YOLO 학습(라벨링 비용 큼) → 제외. PaddleOCR/PP-Structure(무거운 PaddlePaddle 의존, py3.14 불확실) → 제외.
- **효과(설계 의도)**: 페이지 전체 임베딩은 도면+표+본문이 섞여 벡터가 뭉개짐 → 크롭 색인으로 대상만 임베딩해 정밀도↑.

In [ ]:
# (발췌) figure/table 영역 검출 후 크롭
from doclayout_yolo import YOLOv10
from huggingface_hub import hf_hub_download
import numpy as np

w = hf_hub_download("juliozhao/DocLayout-YOLO-DocStructBench",
                    "doclayout_yolo_docstructbench_imgsz1024.pt")
yolo = YOLOv10(w)
TARGET = {"figure": "그림/도면", "table": "표"}

def detect_regions(img, conf=0.25):
    res = yolo.predict(np.array(img.convert("RGB")), imgsz=1024, conf=conf, verbose=False)[0]
    return [(TARGET[yolo.names[int(b.cls)]], b.xyxy[0].tolist())
            for b in res.boxes if yolo.names[int(b.cls)] in TARGET]

### 2-3. OCR — `rapidocr-onnxruntime`

- **역할**: 스캔본 PDF(텍스트 레이어 없음)에서 글자 인식 → 텍스트 검색 가능하게
- **선정 이유**: 한글+영어 지원, **시스템 바이너리 불필요**(pip만으로 설치), **Python 3.14에서 설치·동작 확인**.
- **대안 비교**:
  - **Tesseract**: 별도 실행파일 설치 필요 → 이 환경 제약과 안 맞음
  - **EasyOCR**: 동작은 하나 더 무겁고, 이 프로젝트엔 rapidocr가 더 가벼움
- OCR 신뢰도 임계값으로 **깨진 인식 결과를 필터링**(색인 탭 슬라이더).

### 2-4. 텍스트 임베딩 — `intfloat/multilingual-e5-base` → **`BAAI/bge-m3`** (핵심 의사결정)

한글 질의로 영문 본문을 찾으려면 **교차언어 의미 검색**이 필요합니다. 여기서 두 번의 결정이 있었습니다.

**결정 ①: 키워드 매칭 → 의미 임베딩**
초기엔 텍스트를 키워드(글자 일치)로 검색했는데, 영문 매뉴얼에 **한글로 검색하면 0건**이었습니다(글자가 다르니 당연). → 의미 임베딩으로 전환.

**결정 ②: 번역(opus-mt) 방식은 폐기 → 다국어 임베딩(e5) 채택**
"한글 질의를 영어로 번역 후 검색"도 후보였으나, 번역 모델(Marian/opus-mt)은 **`sentencepiece`** 가 필요한데
**Python 3.14용 휠이 없고**(최신도 cp313까지) 이 PC는 네이티브 빌드가 안 됩니다. → 번역 경로 **폐기**, 다국어 임베딩으로 우회.

**결정 ③: e5-base → bge-m3 교체 (실측 기반)**
다국어 e5-base를 먼저 썼으나, **한글→영문 교차언어 점수가 좁은 구간(0.76~0.80)에 뭉쳐** 관련/무관 구분이 약했습니다.
교차언어 검색 특화인 **bge-m3** 로 교체하니 질의마다 정답 페이지가 뚜렷이 분리됐습니다. (아래 3절 실측)

- `sentencepiece` 불필요(fast tokenizer)라 py3.14에서도 문제없음 → bge-m3 채택.

In [ ]:
# (발췌) 텍스트 임베딩: bge-m3, CLS pooling + L2 정규화 (질의/문서 접두사 불필요)
from transformers import AutoTokenizer, AutoModel
import torch

TEXT_MODEL_ID = "BAAI/bge-m3"
tok = AutoTokenizer.from_pretrained(TEXT_MODEL_ID)
tmodel = AutoModel.from_pretrained(TEXT_MODEL_ID).eval()

@torch.no_grad()
def embed_text(text):
    enc = tok([text], padding=True, truncation=True, max_length=512, return_tensors="pt")
    cls = tmodel(**enc).last_hidden_state[:, 0]         # CLS pooling
    return torch.nn.functional.normalize(cls, p=2, dim=1).cpu().numpy()[0]

## 3. 실측 비교 결과 (measured)

> 동일한 실제 매뉴얼(영문, Oxygen Analyzer)로 색인 후 **한글 질의**로 측정한 값입니다.

### 3-1. 텍스트 모델: e5-base vs bge-m3 (한글→영문 교차언어)

| 한글 질의 | e5-base (이전) | bge-m3 (채택) |
|---|---|---|
| 산소 분석기 | 상위 0.80 · p1/p11/p29 **뭉침** | **p6 0.599** > p8 0.586 (분리) |
| 센서 교정 | 최고 **0.765** (임계값 0.78 미달→0건) | p28 0.544 |
| 경보 설정 | 최고 **0.776** (거의 미달) | p25 0.531 |
| 전원 연결 | 0.779 | p12 0.541 |
| (영어) oxygen analyzer | 0.876 (영어는 잘 됨) | — |

**해석**: e5-base는 같은 언어(영↔영)는 0.86+로 잘 되지만, **교차언어(한→영)는 0.76~0.80으로 압축**되어
임계값 필터가 한글 질의 대부분을 잘라냈습니다. bge-m3는 점수가 더 벌어져 **질의마다 정답 페이지가 1위**로 잡힙니다.

### 3-2. 청크 크기 효과 (텍스트 색인 단위)

| 청크 크기 | "연소 위험 경고"(combustion hazard) 검색 |
|---|---|
| 큰 청크(≈450자, 한 페이지 1청크) | **0건** (여러 주제 섞여 희석, 임계값 미달) |
| 작은 청크(≈320자) | **0.497 로 검색됨** ✅ |

**해석**: 청크가 크면 특정 경고문이 다른 내용에 희석됩니다. 청크를 작게 나눠 **한 청크=한 주제**에 가깝게 하니
특정 내용 검색 정밀도가 올라갔습니다. (동시에 긴 페이지의 512토큰 잘림 문제도 해결)

## 4. 최종 선정 요약

| 구성요소 | 채택 모델 | 핵심 근거 | 제친 대안 |
|---|---|---|---|
| 이미지 임베딩 | SigLIP2 base | 로컬 멀티모달, 6GB에 가벼움 | ResNet(텍스트검색 불가), CLIP(구버전) |
| 레이아웃 검출 | DocLayout-YOLO | 사전학습, figure/table 제공 | 직접학습(비용), PP-Structure(무거움) |
| OCR | rapidocr-onnxruntime | 한/영, 바이너리 불필요, py3.14 OK | Tesseract(바이너리), EasyOCR(무거움) |
| 텍스트 임베딩 | bge-m3 | 교차언어 분리 우수, sentencepiece 불필요 | e5-base(점수 뭉침), 번역(sentencepiece/py3.14 불가) |

## 5. 파이프라인 & 실행

```
[색인] PDF → 페이지 렌더(PyMuPDF)
           → 도면·표 검출·크롭(DocLayout-YOLO) → 이미지 임베딩(SigLIP2)
           → 본문 추출(get_text/OCR) → 청크 분할 → 텍스트 임베딩(bge-m3)
           → indexes/<세트>/index.npz

[검색] 질의 → 이미지: SigLIP2 텍스트 임베딩 → 코사인 유사도
            → 텍스트: bge-m3 임베딩 → 코사인 유사도(다국어)
            → 임계값 이상 Top-K 표시
```

실행 환경은 CUDA/MPS/CPU를 자동 감지합니다.

In [ ]:
# (발췌) 실행 환경 자동 감지
import torch

def detect_device():
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = detect_device()
print("device:", DEVICE)   # 실행 시 cuda / mps / cpu 중 하나

## 6. 한계와 확장 (RAG)

- **한계**: 의미 검색이라 정확한 수치 값 추출은 아님 · 도면 의미 매칭은 실제 도면 이미지에서 평가 필요 · OCR 품질에 좌우
- **확장(RAG)**: 지금의 **청크 색인 = RAG의 지식베이스**. 여기에 로컬 LLM을 붙여
  "질의 → 관련 청크 검색 → 근거로만 답변 생성(+출처 페이지)"로 확장 가능. 세트별로 지식 범위를 한정할 수 있음.

---

### 재현 방법
```bash
pip install -r requirements.txt
python app.py   # 또는 run.bat (Windows)
```
전체 코드: [`app.py`](app.py)